In [1]:
import random

import asyncio
import nest_asyncio

import networkx as nx


from src.utils import Loader
from src.metrics import utility_gap
from src.diffusion_models import (
    estimate_cascade_by_community,
)

In [2]:
random.seed(42)
nest_asyncio.apply()

In [3]:
from pathlib import Path

paths_to_networks = Path('src/data/synthetic/networks')

## Simple Barbasi-Albert Graph

In [4]:
k = 10  # number of seeds to select
alpha = 0  # inequality-aversion parameter
p = 0.1  # edge activation probability
num_sims = 1000  # number of simulations

In [5]:
from src import kempe_greedy


async def main():
    loader = Loader(max_workers=4)

    loaded_graph = await loader.load(f'{paths_to_networks}/barbasi_albert_1000.pkl')

    seeds = kempe_greedy(
        graph=loaded_graph,
        k=k,
        probability=p,
        num_simulations=num_sims,
    )
    print(f'Final seeds: {seeds}')

    final_frac = estimate_cascade_by_community(
        graph=loaded_graph,
        seeds=seeds,
        probability=0.1,
        num_simulations=500,
    )
    print(f'Expected influenced fraction per community: {str({k: round(v, 2) for k, v in final_frac.items()})}')
    print(f'Utility Gap: {round(utility_gap(final_frac), 2)}')


asyncio.run(main())

Selecting seeds: 100%|██████████| 10/10 [01:11<00:00,  7.13s/it, seeds=10]

Final seeds: {1, 2, 66, 5, 6, 14, 15, 55, 23, 27}
Expected influenced fraction per community: {0: 0.05, 1: 0.14, 2: 0.08, 3: 0.08, 4: 0.08, 5: 0.07, 6: 0.05, 7: 0.05, 8: 0.04, 9: 0.03, 10: 0.06, 11: 0.05, 12: 0.04, 13: 0.05, 14: 0.03, 15: 0.05, 16: 0.02, 17: 0.01}
Utility Gap: 12.44


In [7]:
from src import water_filling_greedy


async def main():
    loader = Loader(max_workers=4)

    loaded_graph = await loader.load(f'{paths_to_networks}/barbasi_albert_1000.pkl')
    costs = nx.get_node_attributes(loaded_graph, 'node_costs')

    total_cost = sum(costs.values())
    budget_fractions = [0.005, 0.01, 0.05, 0.1, 0.3]
    budgets = [f * total_cost for f in budget_fractions]

    for budget in budgets:
        seeds = water_filling_greedy(
            graph=loaded_graph,
            costs=costs,
            budget=budget,
            alpha=0.0,
            probability=0.1,
            num_sims=1000,
        )

        comm = estimate_cascade_by_community(
            graph=loaded_graph,
            seeds=seeds,
            probability=p,
            num_simulations=num_sims,
            random_state=42,
        )
        print(f'Expected influenced fraction per community: {str({k: round(v, 2) for k, v in comm.items()})}')
        print(f'Utility Gap: {round(utility_gap(comm), 2)}')


asyncio.run(main())

Seed Selection: 100%|██████████| 983/983 [00:27<00:00, 35.58it/s, seeds=9, welfare=-2841.8980, budget_used=4.51/5.00, queue_size=909]


Expected influenced fraction per community: {0: 0.04, 1: 0.13, 2: 0.05, 3: 0.05, 4: 0.09, 5: 0.08, 6: 0.07, 7: 0.07, 8: 0.03, 9: 0.02, 10: 0.07, 11: 0.07, 12: 0.07, 13: 0.07, 14: 0.02, 15: 0.03, 16: 0.02, 17: 0.01}
Utility Gap: 12.28


Seed Selection: 100%|██████████| 997/997 [00:35<00:00, 28.36it/s, seeds=16, welfare=-2819.5791, budget_used=9.53/10.00, queue_size=961]


Expected influenced fraction per community: {0: 0.06, 1: 0.12, 2: 0.07, 3: 0.05, 4: 0.08, 5: 0.07, 6: 0.06, 7: 0.06, 8: 0.05, 9: 0.02, 10: 0.06, 11: 0.06, 12: 0.05, 13: 0.05, 14: 0.03, 15: 0.05, 16: 0.04, 17: 0.03}
Utility Gap: 10.06


Seed Selection: 100%|██████████| 1000/1000 [00:36<00:00, 27.37it/s, seeds=28, welfare=-2815.0255, budget_used=49.82/50.00, queue_size=963]


Expected influenced fraction per community: {0: 0.05, 1: 0.12, 2: 0.07, 3: 0.06, 4: 0.08, 5: 0.07, 6: 0.06, 7: 0.06, 8: 0.04, 9: 0.02, 10: 0.06, 11: 0.06, 12: 0.04, 13: 0.06, 14: 0.03, 15: 0.04, 16: 0.03, 17: 0.02}
Utility Gap: 10.0


Seed Selection: 100%|██████████| 1000/1000 [00:37<00:00, 26.67it/s, seeds=32, welfare=-2817.0778, budget_used=99.80/100.00, queue_size=942]


Expected influenced fraction per community: {0: 0.06, 1: 0.12, 2: 0.08, 3: 0.06, 4: 0.08, 5: 0.08, 6: 0.05, 7: 0.06, 8: 0.04, 9: 0.03, 10: 0.06, 11: 0.06, 12: 0.04, 13: 0.05, 14: 0.03, 15: 0.04, 16: 0.03, 17: 0.02}
Utility Gap: 10.46


Seed Selection: 100%|██████████| 1000/1000 [00:37<00:00, 26.73it/s, seeds=120, welfare=-2826.6436, budget_used=299.57/300.00, queue_size=880]

Expected influenced fraction per community: {0: 0.07, 1: 0.11, 2: 0.06, 3: 0.06, 4: 0.09, 5: 0.08, 6: 0.05, 7: 0.06, 8: 0.05, 9: 0.02, 10: 0.06, 11: 0.05, 12: 0.04, 13: 0.07, 14: 0.02, 15: 0.05, 16: 0.03, 17: 0.02}
Utility Gap: 8.63
